# Partie 2 — Corrigé des quatre exercices

Exécuter les cellules dans l'ordre. Les données MNIST sont téléchargées dans `./data` lors de la première exécution. Bibliothèques nécessaires : `torch`, `torchvision`, `scikit-learn`, `matplotlib`.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device)


## Exercice 1 — Régression du salaire

Le salaire est simulé par `4000 + 600 × ancienneté + 1500 × expertise + bruit`. On standardise les entrées avec les seules statistiques d'entraînement et on entraîne sur des salaires exprimés en milliers de DH.


In [ ]:
n = 300
anciennete = torch.randint(0, 21, (n, 1)).float()
expertise = torch.randint(1, 6, (n, 1)).float()
bruit = torch.randn(n, 1) * 500
X_salaire = torch.cat([anciennete, expertise], dim=1)
y_salaire = 4000 + 600 * anciennete + 1500 * expertise + bruit

print("Dimensions de X et y :", X_salaire.shape, y_salaire.shape)
print(f"Salaire moyen : {y_salaire.mean().item():.2f} DH")
plt.figure(figsize=(8, 5))
for niveau in range(1, 6):
    masque = X_salaire[:, 1] == niveau
    plt.scatter(X_salaire[masque, 0], y_salaire[masque, 0],
                alpha=0.6, label=f"Expertise {niveau}")
plt.xlabel("Ancienneté (années)")
plt.ylabel("Salaire (DH)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
indices = torch.randperm(n)
limite = int(0.8 * n)
X_train_sal, X_test_sal = X_salaire[indices[:limite]], X_salaire[indices[limite:]]
y_train_sal = y_salaire[indices[:limite]] / 1000
y_test_sal = y_salaire[indices[limite:]] / 1000
moyenne_sal = X_train_sal.mean(dim=0, keepdim=True)
ecart_type_sal = X_train_sal.std(dim=0, keepdim=True)
X_train_sal_norm = (X_train_sal - moyenne_sal) / ecart_type_sal
X_test_sal_norm = (X_test_sal - moyenne_sal) / ecart_type_sal

modele_sal = nn.Linear(2, 1)
critere_sal = nn.MSELoss()
optimiseur_sal = torch.optim.SGD(modele_sal.parameters(), lr=0.05)
historique_sal = []
modele_sal.train()
for epoch in range(500):
    optimiseur_sal.zero_grad()
    pred_train = modele_sal(X_train_sal_norm)
    perte = critere_sal(pred_train, y_train_sal)
    perte.backward()
    optimiseur_sal.step()
    historique_sal.append(perte.item())
    if (epoch + 1) % 100 == 0:
        print(f"Époque {epoch + 1} | MSE : {perte.item():.4f}")

plt.plot(historique_sal)
plt.xlabel("Époque")
plt.ylabel("MSE (milliers de DH)²")
plt.grid()
plt.show()


In [ ]:
modele_sal.eval()
with torch.no_grad():
    predictions_dh = modele_sal(X_test_sal_norm) * 1000
    salaires_reels_dh = y_test_sal * 1000
    mae_sal = torch.mean(torch.abs(predictions_dh - salaires_reels_dh))
    nouveau_salarie = torch.tensor([[7.0, 4.0]])
    nouveau_norm = (nouveau_salarie - moyenne_sal) / ecart_type_sal
    salaire_predit = modele_sal(nouveau_norm).item() * 1000

# Le modèle prédit des milliers de DH à partir de variables standardisées.
# Revenir aux unités d'origine : w_DH = 1000*w_norm/std et
# b_DH = 1000*(b_norm - somme(w_norm*mean/std)).
avec_unites = 1000 * modele_sal.weight.detach().squeeze(0) / ecart_type_sal.squeeze(0)
biais_dh = 1000 * (modele_sal.bias.detach().item()
                  - (modele_sal.weight.detach() * moyenne_sal / ecart_type_sal).sum().item())
print(f"MAE test : {mae_sal.item():.2f} DH")
print(f"Salaire prédit (7 ans, expertise 4) : {salaire_predit:.2f} DH (référence : 14200 DH)")
print(f"Coefficients estimés : ancienneté={avec_unites[0].item():.1f}, "
      f"expertise={avec_unites[1].item():.1f}, biais={biais_dh:.1f} DH")


Une année d'ancienneté supplémentaire augmente le salaire prédit d'environ 600 DH, à expertise constante. Le bruit aléatoire empêche la perte de devenir nulle. Pour 40 ans d'ancienneté, le modèle extrapole hors de l'intervalle observé (0–20 ans), donc sa fiabilité est inconnue. Un bruit plus fort augmente généralement la MAE et rend les coefficients estimés moins précis. Traiter l'expertise comme une note numérique suppose un écart salarial identique entre chaque niveau.


## Exercice 2 — Classification Iris

Les quatre colonnes sont les longueurs et largeurs des sépales et des pétales. `CrossEntropyLoss` attend des étiquettes entières de type `torch.long` et des logits bruts. Le `StandardScaler` est ajusté seulement sur l'entraînement pour éviter que le test influence l'apprentissage.


In [ ]:
iris = load_iris()
X_train_iris_np, X_test_iris_np, y_train_iris_np, y_test_iris_np = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)
scaler_iris = StandardScaler()
X_train_iris = torch.tensor(scaler_iris.fit_transform(X_train_iris_np), dtype=torch.float32)
X_test_iris = torch.tensor(scaler_iris.transform(X_test_iris_np), dtype=torch.float32)
y_train_iris = torch.tensor(y_train_iris_np, dtype=torch.long)
y_test_iris = torch.tensor(y_test_iris_np, dtype=torch.long)
print("Dimensions :", X_train_iris.shape, y_train_iris.shape)

def entrainer_iris(reseau, epoques=200):
    critere = nn.CrossEntropyLoss()
    optimiseur = torch.optim.Adam(reseau.parameters(), lr=0.01)
    pertes = []
    reseau.train()
    for _ in range(epoques):
        optimiseur.zero_grad()
        logits = reseau(X_train_iris)
        perte = critere(logits, y_train_iris)
        perte.backward()
        optimiseur.step()
        pertes.append(perte.item())
    reseau.eval()
    with torch.no_grad():
        predictions = reseau(X_test_iris).argmax(dim=1)
    return pertes, predictions

torch.manual_seed(42)
modele_iris = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
pertes_iris, predictions_iris = entrainer_iris(modele_iris)
print(f"Réseau caché — accuracy test : {(predictions_iris == y_test_iris).float().mean().item():.1%}")
plt.plot(pertes_iris)
plt.xlabel("Époque")
plt.ylabel("CrossEntropyLoss")
plt.title("Iris — perte d'entraînement")
plt.grid()
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test_iris_np, predictions_iris.numpy(),
                                        display_labels=iris.target_names)
plt.title("Iris — matrice de confusion")
plt.show()


In [ ]:
torch.manual_seed(42)
modele_iris_lineaire = nn.Linear(4, 3)
_, predictions_iris_lineaire = entrainer_iris(modele_iris_lineaire)
print(f"Modèle sans couche cachée — accuracy test : "
      f"{(predictions_iris_lineaire == y_test_iris).float().mean().item():.1%}")

nouvelle_fleur = torch.tensor(scaler_iris.transform([[5.1, 3.5, 1.4, 0.2]]),
                             dtype=torch.float32)
with torch.no_grad():
    classe_fleur = modele_iris(nouvelle_fleur).argmax(dim=1).item()
print("Espèce prédite :", iris.target_names[classe_fleur])


La comparaison porte sur le même découpage entraînement/test. Le modèle sans couche cachée trace des frontières linéaires ; le réseau avec couche cachée peut représenter des frontières plus complexes. Sur 30 fleurs de test, quelques exemples suffisent à faire varier sensiblement l'accuracy.


## Exercice 3 — MLP sur MNIST

Entraînement sur 5 époques avec des mini-lots. Pour les métriques multiclasses, on utilise la moyenne macro : chaque chiffre a le même poids. Les poids sont sauvegardés puis rechargés dans une nouvelle instance du réseau.


In [ ]:
from torchvision import datasets, transforms

transform_mnist_mlp = transforms.ToTensor()
train_mnist_mlp = datasets.MNIST("./data", train=True, download=True,
                                transform=transform_mnist_mlp)
test_mnist_mlp = datasets.MNIST("./data", train=False, download=True,
                               transform=transform_mnist_mlp)
train_loader_mlp = DataLoader(train_mnist_mlp, batch_size=64, shuffle=True)
test_loader_mlp = DataLoader(test_mnist_mlp, batch_size=256, shuffle=False)

def creer_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(),
                         nn.Linear(128, 10))

torch.manual_seed(42)
modele_mlp = creer_mlp().to(device)
critere_mlp = nn.CrossEntropyLoss()
optimiseur_mlp = torch.optim.Adam(modele_mlp.parameters(), lr=0.001)
for epoch in range(5):
    modele_mlp.train()
    somme_pertes = 0.0
    for images, labels in train_loader_mlp:
        images, labels = images.to(device), labels.to(device)
        optimiseur_mlp.zero_grad()
        outputs = modele_mlp(images)
        perte = critere_mlp(outputs, labels)
        perte.backward()
        optimiseur_mlp.step()
        somme_pertes += perte.item() * images.size(0)
    print(f"MLP époque {epoch + 1}/5 | perte : {somme_pertes / len(train_mnist_mlp):.4f}")


In [ ]:
modele_mlp.eval()
vrais_mlp, predits_mlp, erreurs_mlp = [], [], []
with torch.no_grad():
    for images, labels in test_loader_mlp:
        outputs = modele_mlp(images.to(device))
        predicted_classes = outputs.argmax(dim=1).cpu()
        vrais_mlp.extend(labels.tolist())
        predits_mlp.extend(predicted_classes.tolist())
        for image, vrai, predit in zip(images, labels, predicted_classes):
            if vrai.item() != predit.item() and len(erreurs_mlp) < 10:
                erreurs_mlp.append((image.squeeze(0).clone(), vrai.item(), predit.item()))

print(f"MLP accuracy  : {accuracy_score(vrais_mlp, predits_mlp):.4f}")
print(f"MLP precision : {precision_score(vrais_mlp, predits_mlp, average='macro', zero_division=0):.4f}")
print(f"MLP recall    : {recall_score(vrais_mlp, predits_mlp, average='macro', zero_division=0):.4f}")
print(f"MLP F1-score  : {f1_score(vrais_mlp, predits_mlp, average='macro', zero_division=0):.4f}")
ConfusionMatrixDisplay.from_predictions(vrais_mlp, predits_mlp, labels=list(range(10)),
                                        cmap="Blues", values_format="d")
plt.title("MLP — matrice de confusion")
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for ax, (image, vrai, predit) in zip(axes.flat, erreurs_mlp):
    ax.imshow(image.numpy(), cmap="gray")
    ax.set_title(f"Vrai {vrai} / Prédit {predit}")
    ax.axis("off")
for ax in list(axes.flat)[len(erreurs_mlp):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

torch.save(modele_mlp.state_dict(), "mnist_mlp.pth")
modele_mlp_recharge = creer_mlp().to(device)
modele_mlp_recharge.load_state_dict(torch.load("mnist_mlp.pth", map_location=device,
                                               weights_only=True))
modele_mlp_recharge.eval()
with torch.no_grad():
    lot_controle = next(iter(test_loader_mlp))[0].to(device)
    assert torch.allclose(modele_mlp(lot_controle), modele_mlp_recharge(lot_controle))
print("Modèle MLP rechargé : sorties identiques sur un lot de test.")


## Exercice 4 — CNN sur MNIST

Deux convolutions 3×3 avec padding 1 conservent la hauteur et la largeur ; deux max-poolings 2×2 réduisent 28×28 à 14×14 puis 7×7. Avec 32 canaux en sortie, la première couche dense reçoit donc `32 × 7 × 7` valeurs. Le dropout est placé avant la dernière couche linéaire.


In [ ]:
transform_mnist_cnn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_mnist_cnn = datasets.MNIST("./data", train=True, download=True,
                                transform=transform_mnist_cnn)
test_mnist_cnn = datasets.MNIST("./data", train=False, download=True,
                               transform=transform_mnist_cnn)
train_loader_cnn = DataLoader(train_mnist_cnn, batch_size=64, shuffle=True)
test_loader_cnn = DataLoader(test_mnist_cnn, batch_size=256, shuffle=False)
images_exemple, labels_exemple = next(iter(train_loader_cnn))
print("Images :", images_exemple.shape, "Étiquettes :", labels_exemple.shape)

fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i, ax in enumerate(axes):
    image = images_exemple[i, 0] * 0.3081 + 0.1307
    ax.imshow(image.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Classe : {labels_exemple[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


Les quatre dimensions d'un lot d'images sont `[nombre d'images, canaux, hauteur, largeur]`. `shuffle=True` change l'ordre des exemples à chaque époque et évite des mini-lots systématiquement biaisés par l'ordre des données.


In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(42)
modele_cnn = MNISTClassifier().to(device)
with torch.no_grad():
    assert modele_cnn(images_exemple.to(device)).shape == (images_exemple.size(0), 10)
print("Forme de sortie CNN vérifiée.")

critere_cnn = nn.CrossEntropyLoss()
optimiseur_cnn = torch.optim.Adam(modele_cnn.parameters(), lr=0.001)
historique_cnn = []
for epoch in range(3):
    modele_cnn.train()
    somme_pertes = 0.0
    for images, labels in train_loader_cnn:
        images, labels = images.to(device), labels.to(device)
        optimiseur_cnn.zero_grad()
        logits = modele_cnn(images)
        perte = critere_cnn(logits, labels)
        perte.backward()
        optimiseur_cnn.step()
        somme_pertes += perte.item() * images.size(0)
    historique_cnn.append(somme_pertes / len(train_mnist_cnn))
    print(f"CNN époque {epoch + 1}/3 | perte : {historique_cnn[-1]:.4f}")


In [ ]:
modele_cnn.eval()
correct, total = 0, 0
erreurs_cnn = []
with torch.no_grad():
    for images, labels in test_loader_cnn:
        logits = modele_cnn(images.to(device))
        predictions = logits.argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        for image, vrai, predit in zip(images, labels, predictions):
            if vrai.item() != predit.item() and len(erreurs_cnn) < 10:
                erreurs_cnn.append((image.squeeze(0).clone(), vrai.item(), predit.item()))

print(f"CNN accuracy test : {correct / total:.2%}")
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for ax, (image, vrai, predit) in zip(axes.flat, erreurs_cnn):
    ax.imshow(image.numpy() * 0.3081 + 0.1307, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Vrai {vrai} / Prédit {predit}")
    ax.axis("off")
for ax in list(axes.flat)[len(erreurs_cnn):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

torch.save(modele_cnn.state_dict(), "mnist_cnn.pth")
print("Paramètres du CNN sauvegardés dans mnist_cnn.pth")
